### 🧠 Part A — Bulletproof Error Handling

Below are 3 programs from typical Day 7–10 topics refactored with:

try

except (specific exceptions)

else

finally

raise for validation

logging

### Program 1 — Safe Age Calculator (Input Validation)

In [1]:
import logging

logging.basicConfig(filename="errors.log", level=logging.ERROR)

while True:
    try:
        age = int(input("Enter your age: "))

        if age < 0 or age > 150:
            raise ValueError(f"Age must be between 0 and 150. Got {age}")

    except ValueError as e:
        print(f"Invalid input: {e}")
        logging.error(f"Age input error: {e}")

    else:
        print(f"In 10 years you will be {age + 10}")
        break

    finally:
        print("Attempt completed.\n")

Enter your age:  23


In 10 years you will be 33
Attempt completed.



### Program 2 — Dictionary Lookup Program

In [ ]:
students = {
    "rahul": 85,
    "aman": 92,
    "neha": 88
}

while True:
    try:
        name = input("Enter student name: ").lower()

        if not name.isalpha():
            raise ValueError("Name must contain only letters")

        marks = students[name]

    except KeyError:
        print("Student not found in database")

    except ValueError as e:
        print(e)

    else:
        print(f"{name.title()} scored {marks}")
        break

    finally:
        print("Lookup attempt finished\n")

Enter student name:  23


Name must contain only letters
Lookup attempt finished



Enter student name:  23


Name must contain only letters
Lookup attempt finished



Enter student name:  453


Name must contain only letters
Lookup attempt finished



### Program 3 — List Average Calculator

In [ ]:
import logging

logging.basicConfig(filename="calc_errors.log", level=logging.ERROR)

try:
    numbers = input("Enter numbers separated by space: ")

    values = numbers.split()

    if len(values) == 0:
        raise ValueError("No numbers entered")

    nums = [float(n) for n in values]

except ValueError as e:
    print(f"Invalid number detected: {e}")
    logging.error(f"Conversion error: {e}")

else:
    avg = sum(nums) / len(nums)
    print(f"Average = {avg}")

finally:
    print("Program finished.")

### Error Handling Checklist

## Program 1 — Age Calculator

Exceptions Caught

* ValueError

Recovery Action

* Ask the user to re-enter the value

User Message
"Invalid input: Age must be between 0 and 150"

Internal Logging
Errors logged to `errors.log`

---

## Program 2 — Student Lookup

Exceptions Caught

* KeyError
* ValueError

Recovery Action

* Inform user the name does not exist
* Ask for correct input

User Message
"Student not found in database"

Internal Logging
No logging required for this program

---

## Program 3 — Average Calculator

Exceptions Caught

* ValueError

Recovery Action

* Stop program safely
* Inform user about invalid numbers

User Message
"Invalid number detected"

Internal Logging
Errors logged to `calc_errors.log`


### 🚀 Part B — Resilient File Processor

In [ ]:
import csv
import json
import time
import traceback
from pathlib import Path

directory = Path("data")

report = {
    "files_processed": 0,
    "files_failed": 0,
    "error_details": {}
}

for file in directory.glob("*.csv"):

    attempts = 0

    while attempts < 3:

        try:
            with open(file, "r", newline="") as f:
                reader = csv.DictReader(f)

                total = 0
                rows = 0

                for row in reader:
                    total += float(row.get("price", 0))
                    rows += 1

            report["files_processed"] += 1
            break

        except PermissionError as e:
            attempts += 1
            time.sleep(1)

            if attempts == 3:
                report["files_failed"] += 1
                report["error_details"][file.name] = str(e)

        except Exception:
            report["files_failed"] += 1
            report["error_details"][file.name] = traceback.format_exc()
            break

with open("processing_report.json", "w") as f:
    json.dump(report, f, indent=4)

### 💼 Part C — Interview Ready
Q1 — Execution Flow
try

Code that might raise an exception is placed here.

except

Runs only if an exception occurs.

else

Runs only if the try block succeeds with no errors.

finally

Runs no matter what, even if an exception occurs.

In [ ]:
try:
    x = int(input("Enter number: "))
except ValueError:
    print("Invalid number")
else:
    print(f"Square = {x*x}")
finally:
    print("Execution finished")

### Q2 — safe_json_load()

In [ ]:
import json
import logging

logging.basicConfig(filename="json_errors.log", level=logging.ERROR)

def safe_json_load(filepath):

    try:
        with open(filepath, "r") as f:
            data = json.load(f)

    except FileNotFoundError as e:
        logging.error(f"File not found: {e}")
        return None

    except json.JSONDecodeError as e:
        logging.error(f"Invalid JSON: {e}")
        return None

    except PermissionError as e:
        logging.error(f"Permission denied: {e}")
        return None

    else:
        return data

### Q3 — Fixed Code

In [ ]:
def process_data(data_list):

    results = []

    for item in data_list:

        try:
            value = int(item)
            results.append(value * 2)

        except ValueError as e:
            print(f"Invalid item '{item}': {e}")
            continue

    return results

### 🤖 Part D — AI Decorator

### Write a Python decorator called retry(max_attempts=3, delay=1)
that retries a function when it raises an exception.

Use exponential backoff and preserve function metadata
using functools.wraps.

In [ ]:
import time
import functools

def retry(max_attempts=3, delay=1):

    def decorator(func):

        @functools.wraps(func)
        def wrapper(*args, **kwargs):

            attempts = 0

            while attempts < max_attempts:

                try:
                    return func(*args, **kwargs)

                except Exception as e:
                    attempts += 1

                    if attempts == max_attempts:
                        raise e

                    wait = delay * (2 ** (attempts - 1))
                    time.sleep(wait)

        return wrapper

    return decorator

In [ ]:
import random

@retry(max_attempts=3, delay=1)
def unstable_function():

    if random.random() < 0.5:
        raise ValueError("Random failure")

    return "Success"

print(unstable_function())

### Critical Evaluation (200 words)

The AI-generated retry decorator successfully implements automatic retry functionality using a wrapper function and exponential backoff. The decorator structure is correct and uses nested functions to allow parameterization of retry behavior. A particularly strong aspect of the implementation is the use of functools.wraps, which preserves the original function's metadata such as its name and docstring. This is important in real-world systems where introspection, logging, or documentation tools depend on accurate function metadata.

The retry logic itself is also implemented properly. When an exception occurs, the decorator increases the attempt counter and waits using exponential backoff before retrying the function. This approach is commonly used in distributed systems to avoid overwhelming external services.

However, the implementation has several limitations. First, it catches a generic Exception, which may retry errors that should not be retried, such as programming bugs or invalid arguments. Ideally, the decorator should allow specifying retryable exception types. Second, it lacks logging, which would be useful for diagnosing failures in production environments. Third, it does not provide hooks for handling final failures gracefully.

To improve the decorator, I would add configurable retryable exception classes, logging support, and optional jitter in the backoff timing to reduce retry synchronization problems in distributed systems.